# Tech Challenge Fase 4 - Demo Multimodal

Notebook de demonstracao do M7: audio sintetico, video sintetico e fluxo integrado via LangGraph.

Esta demo usa servicos e artefatos reais. Antes de executar, configure `OPENAI_API_KEY`, instale as dependencias de `requirements.txt`, gere/treine o modelo YOLOv8 da Fase 4 e mantenha o vectorstore/adapter local disponiveis.

## 1. Setup e preflight

A celula abaixo valida os pre-requisitos da demo. Se algo estiver ausente, execute os milestones anteriores antes de continuar.

In [ ]:
from __future__ import annotations

import importlib
import os
import sys
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import Audio, Markdown, display

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent
sys.path.insert(0, str(BASE_DIR))

load_dotenv(BASE_DIR / ".env")

from src.config import DEFAULT_YOLO_MODEL_PATH, FINETUNE_FINAL_ADAPTER_DIR, VECTORSTORE_DIR

DEMO_DIR = BASE_DIR / "artifacts" / "demo_phase4"
AUDIO_DIR = DEMO_DIR / "audio"
VIDEO_DIR = DEMO_DIR / "video"
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_DIR.mkdir(parents=True, exist_ok=True)

missing = []
if not os.getenv("OPENAI_API_KEY"):
    missing.append("OPENAI_API_KEY nao configurada no ambiente ou .env")
if not DEFAULT_YOLO_MODEL_PATH.exists():
    missing.append(
        f"Pesos YOLOv8 ausentes em {DEFAULT_YOLO_MODEL_PATH}. "
        "Execute scripts/generate_synthetic_data.py e scripts/train_yolo.py."
    )
if not (VECTORSTORE_DIR / "chroma.sqlite3").exists():
    missing.append(f"Vectorstore ausente em {VECTORSTORE_DIR}. Execute scripts/build_kb.py.")
if not FINETUNE_FINAL_ADAPTER_DIR.exists():
    missing.append(f"Adapter local ausente em {FINETUNE_FINAL_ADAPTER_DIR}. Execute scripts/fine_tune.py.")
if not (BASE_DIR / "data" / "patients" / "P-0001.json").exists():
    missing.append("Registro sintetico data/patients/P-0001.json ausente")

for module_name in ("openai", "ultralytics", "cv2", "langgraph", "langchain_core"):
    if importlib.util.find_spec(module_name) is None:
        missing.append(f"Dependencia Python ausente: {module_name}")

if missing:
    raise RuntimeError("Preflight falhou:\n- " + "\n- ".join(missing))

display(Markdown("Preflight concluido. Artefatos e credenciais principais estao disponiveis."))
print(f"Demo dir: {DEMO_DIR}")
print(f"YOLO model: {DEFAULT_YOLO_MODEL_PATH}")

## 2. Pipeline de audio

Geramos um audio clinico sintetico via OpenAI TTS. O audio abaixo e artificial e nao representa uma paciente real.

In [ ]:
from openai import OpenAI

AUDIO_PATH = AUDIO_DIR / "consulta_pos_parto_sintetica.mp3"

synthetic_audio_text = (
    "Eu tive meu bebe ha tres semanas e desde entao nao consigo dormir direito. "
    "Choro varias vezes por dia, sinto muita culpa e tenho medo de nao dar conta. "
    "Meu companheiro fica irritado quando peco ajuda e as vezes eu evito falar para nao piorar a situacao. "
    "Tambem sinto o corpo exausto, com muita fadiga e ansiedade constante."
)

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
with client.audio.speech.with_streaming_response.create(
    model="gpt-4o-mini-tts",
    voice="coral",
    input=synthetic_audio_text,
    instructions="Fale em portugues brasileiro, com tom natural, cansado e cuidadoso.",
) as response:
    response.stream_to_file(AUDIO_PATH)

print(f"Audio sintetico salvo em: {AUDIO_PATH}")
display(Audio(str(AUDIO_PATH)))

In [ ]:
from src.multimodal.audio import analyze_transcript, transcribe

audio_transcript = transcribe(AUDIO_PATH, model="whisper-1")
audio_analysis = analyze_transcript(audio_transcript)

display(Markdown("### Transcript"))
print(audio_transcript)
display(Markdown("### Laudo clinico estruturado"))
print(audio_analysis)

## 3. Pipeline de video

Geramos um video curto a partir do mesmo gerador de frames sinteticos usado no dataset YOLOv8, depois executamos inferencia frame a frame com o modelo treinado.

In [ ]:
import cv2
import numpy as np

from scripts.generate_synthetic_data import generate_frame

VIDEO_PATH = VIDEO_DIR / "sangramento_sintetico.mp4"
IMAGE_SIZE = 640
FPS = 4
FRAME_COUNT = 16

rng = np.random.default_rng(42)
writer = cv2.VideoWriter(
    str(VIDEO_PATH),
    cv2.VideoWriter_fourcc(*"mp4v"),
    FPS,
    (IMAGE_SIZE, IMAGE_SIZE),
)
if not writer.isOpened():
    raise RuntimeError("Nao foi possivel abrir o VideoWriter do OpenCV para gerar o MP4 sintetico.")

try:
    for _ in range(FRAME_COUNT):
        frame_rgb, _boxes = generate_frame(IMAGE_SIZE, rng, min_blobs=1, max_blobs=3)
        writer.write(cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR))
finally:
    writer.release()

print(f"Video sintetico salvo em: {VIDEO_PATH}")
display(Markdown(f"Arquivo de video: `{VIDEO_PATH}`"))

In [ ]:
from src.multimodal.video import analyze_video, generate_video_report

video_analysis = analyze_video(
    VIDEO_PATH,
    DEFAULT_YOLO_MODEL_PATH,
    confidence_threshold=0.25,
    frame_stride=1,
)
video_report = generate_video_report(video_analysis.detections)

display(Markdown("### Resultado tecnico"))
print(f"Frames processados: {video_analysis.frames_processed}")
print(f"Deteccoes: {len(video_analysis.detections)}")
display(Markdown("### Relatorio de video"))
print(video_report)

## 4. Fluxo integrado via LangGraph

Agora executamos o grafo completo com pergunta clinica, contexto de paciente sintetico, audio e video.

In [ ]:
from src.assistant.graph import build_graph

query = (
    "Considerando oncologia, saude materna no pos-parto e achados visuais de sangramento, "
    "quais riscos devem ser priorizados e quais proximos passos clinicos sao recomendados?"
)

state = {
    "query": query,
    "patient_id": "P-0001",
    "intent": "",
    "retrieved_docs": [],
    "kb_context": "",
    "patient_record": None,
    "patient_context": "",
    "answer": "",
    "kb_sources": [],
    "patient_source": None,
    "used_patient_context": False,
    "refused": False,
    "error": None,
    "audio_path": str(AUDIO_PATH),
    "audio_transcript": None,
    "audio_analysis": None,
    "video_path": str(VIDEO_PATH),
    "video_report": None,
}

graph = build_graph()
result = graph.invoke(state)

display(Markdown("### Resposta final do assistente"))
print(result["answer"])

display(Markdown("### Artefatos multimodais no estado final"))
print("Transcript de audio:")
print(result.get("audio_transcript"))
print("\nLaudo de audio:")
print(result.get("audio_analysis"))
print("\nRelatorio de video:")
print(result.get("video_report"))

## 5. Criterios observaveis da demo

- Audio sintetico criado em `artifacts/demo_phase4/audio/`.
- Transcript gerado por Whisper.
- Laudo clinico de audio com categorias de risco.
- Video sintetico criado em `artifacts/demo_phase4/video/`.
- Relatorio de video com total de deteccoes, confidence medio e risco.
- Resposta final do LangGraph com fontes de audio e video.